# 3D Cell Segmentation Pipeline
**Leica `.lif` confocal microscopy — morphological analysis**

Pipeline steps:
1. Mount Drive and load file
2. Inspect file structure
3. Visualize channels
4. Segment cells with Cellpose
5. Compute 3D metrics across all tiles
6. Validate results
7. Plot and export

In [ ]:
%pip install -q readlif cellpose scikit-image matplotlib numpy scipy pandas tifffile


## 1. Imports and Setup

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from readlif.reader import LifFile
from cellpose import models
from skimage.measure import regionprops, marching_cubes, mesh_surface_area
from skimage.exposure import rescale_intensity
from scipy.stats import gaussian_kde

%matplotlib inline
plt.rcParams.update({
    'figure.facecolor': '#111111',
    'axes.facecolor':   '#111111',
    'text.color':       'white',
    'axes.titlecolor':  'white',
})

def norm(img, p_low=1, p_high=99.5):
    lo, hi = np.percentile(img, p_low), np.percentile(img, p_high)
    return np.clip((img.astype(np.float32) - lo) / (hi - lo + 1e-8), 0, 1)

print('✅ Ready')

## 2. Mount Drive and Load File

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

LIF_FILE    = "/content/drive/MyDrive/Neil.lif"
OUTPUT_DIR  = "/content/drive/MyDrive/"

lif         = LifFile(LIF_FILE)
all_series  = list(lif.get_iter_image())

print(f"File: {LIF_FILE}")
print(f"Found {len(all_series)} conditions\n")
print(f"{'#':<5} {'Name':<10} {'W':>6} {'H':>6} {'m':>6} {'Z':>6}")
print("-" * 40)
for i, img in enumerate(all_series):
    print(f"{i:<5} {img.name:<10} {img.dims.x:>6} {img.dims.y:>6} "
          f"{img.dims.m:>6} {img.dims.z:>6}")

## 3. Inspect File Metadata and Visualize Channels

In [ ]:
img_obj   = all_series[0]
scale     = img_obj.scale
voxel_xy  = 1 / scale[0]
voxel_z   = 1 / scale[2]
voxel_vol = voxel_xy * voxel_xy * voxel_z
bit_depth = img_obj.bit_depth[0]

print(f"Voxel XY   : {voxel_xy:.4f} µm/px")
print(f"Voxel Z    : {voxel_z:.4f} µm/slice")
print(f"Voxel vol  : {voxel_vol:.4f} µm³")
print(f"Bit depth  : {bit_depth}-bit  →  max = {2**bit_depth - 1}")
print(f"Noise thr  : {0.04 * (2**bit_depth - 1):.1f}")

In [ ]:
# Find best Z and visualize both channels for condition 0
z_means = [np.array(img_obj.get_frame(z=z, t=0, c=1, m=0)).mean()
           for z in range(img_obj.dims.z)]
best_z  = int(np.argmax(z_means))

ch0 = np.array(img_obj.get_frame(z=best_z, t=0, c=0, m=0))
ch1 = np.array(img_obj.get_frame(z=best_z, t=0, c=1, m=0))

fig, axes = plt.subplots(1, 2, figsize=(14, 7))
fig.suptitle(f"'{img_obj.name}' — Z={best_z} (brightest)", fontsize=13)
axes[0].imshow(norm(ch0), cmap='Blues_r');  axes[0].set_title('Ch0 — Alexa 405 (organelles)'); axes[0].axis('off')
axes[1].imshow(norm(ch1), cmap='Greens_r'); axes[1].set_title('Ch1 — Alexa 488 (membrane)');  axes[1].axis('off')
plt.tight_layout(); plt.show()

## 4. Load Cellpose Model

In [ ]:
model = models.CellposeModel(gpu=True)
print('✅ Cellpose ready')

## 5. Full Pipeline — All Conditions, All Tiles

For each condition and tile:
- Auto-detect best Z from Ch1 signal
- Run Cellpose 2D on best Z
- Reconstruct 3D volume using FWHM criterion (50% of peak per cell)
- Compute all 3D metrics
- Save results to Drive after each tile

In [ ]:
save_path = os.path.join(OUTPUT_DIR, 'cell_results_all_tiles.csv')
if os.path.exists(save_path):
    os.remove(save_path)

for img_obj in all_series:
    condition = img_obj.name
    scale     = img_obj.scale
    voxel_xy  = 1 / scale[0]
    voxel_z   = 1 / scale[2]
    voxel_vol = voxel_xy * voxel_xy * voxel_z
    n_z       = img_obj.dims.z
    n_tiles   = img_obj.dims.m
    bit_depth = img_obj.bit_depth[0]
    ch0_thr   = 0.04 * (2**bit_depth - 1)

    print(f"\n{'='*55}")
    print(f"Condition: {condition}  |  Tiles: {n_tiles}  |  Z: {n_z}")
    print(f"{'='*55}")

    # Auto-detect best Z
    z_means = [np.array(img_obj.get_frame(z=z, t=0, c=1, m=0)).mean()
               for z in range(n_z)]
    best_z  = int(np.argmax(z_means))
    z_start = max(0, best_z - 10)
    z_end   = min(n_z, best_z + 10)
    print(f"Best Z: {best_z}  |  Window: Z={z_start}–{z_end}")

    for tile_idx in range(n_tiles):
        print(f"\n  Tile {tile_idx+1:>2}/{n_tiles} ...", end=' ', flush=True)

        # Load stacks
        stack_ch1 = np.stack([np.array(img_obj.get_frame(z=z, t=0, c=1, m=tile_idx))
                               for z in range(z_start, z_end)])
        stack_ch0 = np.stack([np.array(img_obj.get_frame(z=z, t=0, c=0, m=tile_idx))
                               for z in range(z_start, z_end)])

        # Normalize for Cellpose
        best_z_local = int(np.argmax([s.mean() for s in stack_ch1]))
        frame_norm   = rescale_intensity(
            stack_ch1[best_z_local].astype(np.float32), out_range=(0, 255)
        ).astype(np.uint8)

        # 2D segmentation
        masks_2d, _, _ = model.eval(frame_norm, diameter=None,
                                     flow_threshold=0.4, cellprob_threshold=0.0)
        print(f"2D: {masks_2d.max()} cells", end=' ', flush=True)

        rows = []
        for region in regionprops(masks_2d):
            cell_mask_2d = masks_2d == region.label

            # Exclude border cells
            bb = region.bbox
            if bb[0]==0 or bb[1]==0: continue
            if bb[2]==masks_2d.shape[0] or bb[3]==masks_2d.shape[1]: continue

            # FWHM — build 3D mask
            z_profile  = np.array([stack_ch1[z][cell_mask_2d].mean()
                                    for z in range(stack_ch1.shape[0])])
            active_z   = np.where(z_profile >= z_profile.max() * 0.5)[0]
            if len(active_z) < 3: continue

            cell_3d = np.zeros_like(stack_ch1, dtype=bool)
            for z in active_z:
                cell_3d[z] = cell_mask_2d

            # 3D metrics
            volume_um3 = cell_3d.sum() * voxel_vol

            try:
                verts, faces, _, _ = marching_cubes(
                    cell_3d.astype(np.uint8), level=0.5,
                    spacing=(voxel_z, voxel_xy, voxel_xy))
                surface_um2 = mesh_surface_area(verts, faces)
                sphericity  = min((np.pi**(1/3) * (6*volume_um3)**(2/3)) / surface_um2, 1.0)
            except Exception:
                surface_um2 = np.nan
                sphericity  = np.nan

            feret_max    = region.axis_major_length * voxel_xy
            feret_min    = region.axis_minor_length * voxel_xy
            feret_ratio  = feret_min / (feret_max + 1e-8)
            ch0_vals     = stack_ch0[cell_3d]
            ch0_vol_ratio = float((ch0_vals > ch0_thr).sum()) * voxel_vol / (volume_um3 + 1e-8)

            rows.append({
                'condition':      condition,
                'tile':           tile_idx,
                'cell_id':        region.label,
                'volume_um3':     round(volume_um3, 2),
                'surface_um2':    round(surface_um2, 2) if not np.isnan(surface_um2) else np.nan,
                'sphericity':     round(sphericity, 4)  if not np.isnan(sphericity)  else np.nan,
                'feret_max_um':   round(feret_max, 2),
                'feret_min_um':   round(feret_min, 2),
                'feret_ratio':    round(feret_ratio, 4),
                'ch1_mean':       round(float(stack_ch1[cell_3d].mean()), 2),
                'ch0_mean':       round(float(ch0_vals.mean()), 2),
                'ch0_total':      round(float(ch0_vals.sum()), 2),
                'ch0_volume_um3': round(float((ch0_vals > ch0_thr).sum()) * voxel_vol, 2),
                'ch0_vol_ratio':  round(ch0_vol_ratio, 4),
                'cell_height_um': round(len(active_z) * voxel_z, 2),
                'n_active_z':     len(active_z),
            })

        if rows:
            df_tile      = pd.DataFrame(rows)
            write_header = not os.path.exists(save_path)
            df_tile.to_csv(save_path, mode='a', header=write_header, index=False)
            print(f"→ {len(rows)} saved")
        else:
            print("→ 0 kept")

df = pd.read_csv(save_path)
print(f"\n✅ DONE — Total cells: {len(df)}")
print(df.groupby('condition')[['volume_um3','sphericity','feret_ratio','ch0_vol_ratio']].mean().round(3))

## 6. Results and Visualisation

In [ ]:
df = pd.read_csv(save_path)

metrics = {
    'volume_um3':     'Volume (µm³)',
    'surface_um2':    'Surface Area (µm²)',
    'sphericity':     'Sphericity',
    'feret_max_um':   'Feret Max (µm)',
    'feret_ratio':    'Feret Ratio (min/max)',
    'ch0_mean':       'Ch0 Mean Intensity',
    'ch0_vol_ratio':  'Ch0 / Cell Volume Ratio',
    'cell_height_um': 'Cell Height (µm)',
}

conditions = ['WT', 'N1', 'N2', 'N3']
colors     = {'WT':'#4CAF50', 'N1':'#2196F3', 'N2':'#FF9800', 'N3':'#E91E63'}

fig, axes = plt.subplots(2, 4, figsize=(22, 10))
fig.suptitle('3D Cell Morphology — WT vs N1 vs N2 vs N3', fontsize=15, y=1.01)
axes = axes.flatten()

for ax, (col, label) in zip(axes, metrics.items()):
    data = [df[df.condition==c][col].dropna().values for c in conditions]
    bp   = ax.boxplot(data, patch_artist=True, widths=0.5,
                      medianprops=dict(color='white', linewidth=2),
                      flierprops=dict(marker='.', markersize=3, alpha=0.3))
    for patch, cond in zip(bp['boxes'], conditions):
        patch.set_facecolor(colors[cond]); patch.set_alpha(0.8)
    for i, (cond, vals) in enumerate(zip(conditions, data)):
        ax.scatter(np.random.normal(i+1, 0.07, len(vals)), vals,
                   alpha=0.3, s=8, color=colors[cond], zorder=5)
    ax.set_xticklabels(conditions, fontsize=11)
    ax.set_title(label, fontsize=11, pad=6)
    ax.set_facecolor('#1a1a1a')
    ax.tick_params(colors='white')
    for spine in ax.spines.values(): spine.set_edgecolor('#444')

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'cell_morphology_final.png'),
            dpi=150, bbox_inches='tight', facecolor='#111111')
plt.show()

## 7. Validation

In [ ]:
r = 7.5  # 15µm diameter sphere
V_theory = (4/3) * np.pi * r**3
A_theory = 4 * np.pi * r**2
S_theory = (np.pi**(1/3) * (6*V_theory)**(2/3)) / A_theory

print("Theoretical perfect sphere (d=15µm):")
print(f"  Volume      : {V_theory:.1f} µm³")
print(f"  Surface area: {A_theory:.1f} µm²")
print(f"  Sphericity  : {S_theory:.3f}")

print("\nMeasured WT means:")
wt = df[df.condition=='WT']
print(f"  Volume      : {wt.volume_um3.mean():.1f} µm³")
print(f"  Surface area: {wt.surface_um2.mean():.1f} µm²")
print(f"  Sphericity  : {wt.sphericity.mean():.3f}")
print(f"  Feret max   : {wt.feret_max_um.mean():.1f} µm")

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(22, 5))
fig.suptitle('Volume per tile — consistency check', fontsize=13)

for ax, cond in zip(axes, ['WT','N1','N2','N3']):
    dc = df[df.condition==cond]
    tm = dc.groupby('tile').volume_um3.mean()
    ts = dc.groupby('tile').volume_um3.std()
    ax.bar(tm.index, tm.values, yerr=ts.values,
           color=colors[cond], alpha=0.8, capsize=3)
    ax.axhline(tm.mean(), color='white', linestyle='--', linewidth=1.5)
    ax.set_title(f'{cond}'); ax.set_xlabel('Tile'); ax.set_ylabel('Mean volume (µm³)')

plt.tight_layout(); plt.show()

## 8. Export Final Results

In [ ]:
# Summary table
summary = df.groupby('condition')[list(metrics.keys())].agg(['mean','std']).round(3)
summary.columns = [f'{col}_{stat}' for col, stat in summary.columns]
summary = summary.reindex(['WT','N1','N2','N3'])
summary.to_csv(os.path.join(OUTPUT_DIR, 'cell_morphology_summary.csv'))

print(" Files saved to Drive:")
print(f"   cell_results_all_tiles.csv   — {len(df)} cells")
print(f"   cell_morphology_summary.csv  — mean ± std per condition")
print(f"   cell_morphology_final.png    — boxplot figure")
print(f"\nCells per condition:")
print(df.groupby('condition').size())